# Assignment and Alignment of libraries

In [1]:
import os
import pandas as pd
import glob
import sys
genomeFiles = os.path.join('Database','DMS_templates')
sequenceDirectory = os.path.join('Sequencing','DMS')
segments = ['PB2','PB1','PA']
ampliconDesign = os.path.join(genomeFiles, 'amplicon_design.csv')
alignmentDirectory = os.path.join('Results','wholeGenomeAlign')
if not os.path.isdir(alignmentDirectory):
    os.mkdir(alignmentDirectory)
endMatching = 15
scripts = 'Scripts'
sys.path.append(scripts)
overwrite = False
import mutCalling
import importlib
#multiprocessing for mapping to speed things up a bit
from multiprocessing import Pool
numProcessors =2
qvalueCutoff = 30
import importlib
import numpy as np
specialcRNA ={'depleted':['PB2','PA'], 'enriched':['PB2','PA']}
trim3prime = {}

## Make STAR indices for all subamplicons, RACE, and plasmid samples

First, if they have not yet been generated, we should go ahead and make all STAR indices

In [12]:
for segment in segments:
    for amplicon in glob.glob(os.path.join(os.path.join(genomeFiles, segment), '*.fa')):
        folder = amplicon.split('.')[0]
        if not os.path.isdir(folder):
            store = !STAR --runMode genomeGenerate --runThreadN 1 --genomeDir $folder --genomeFastaFiles $amplicon --genomeSAindexNbases 3 


## Separate amplicons

For each dataset, run a script that separates the two fastq files (R1 and R2) into constituent fastq files for each amplicon, and then runs alignments for each and separates each. This enables good template alignment rather than relying on STAR to sort things out. A bit of a cost in terms of computational time, but I think well worth it.

In [2]:
importlib.reload(mutCalling)

def analyzeSegment(arguments):
    overwrite = arguments[1]
    segment = arguments[0]
    if overwrite: 
        remove = os.path.join(segment, '*.fastq*')
        !rm -rf $remove
    #still get rid of any intermediate files regardless
    remove = os.path.join(segment, '*.fastq')
    !rm -rf $remove
    preexisting = [os.path.basename(element).split('_')[0] for element in glob.glob(os.path.join(segment, "*.fastq.gz"))]
    
    for run in glob.glob(os.path.join(segment, '*')):
        files = pd.DataFrame({'file':glob.glob(os.path.join(run, '*.fastq.gz'))})

        if len(files) != 0:
            files[['name','read','stuff']] = files.file.str.split('_', expand=True)
            files['replicate'] = files.name.str[-1:]

            for replicate in set(files.replicate):
                currFrame = files[files.replicate == replicate]
                R1 = currFrame[currFrame.read == 'R1'].file.iloc[0]
                R2 = currFrame[currFrame.read == 'R2'].file.iloc[0]
                baseName = os.path.join(segment, replicate)
                #check if already analyzed and skip if so.
                if replicate not in preexisting:
                        
                    R1unzip = R1[:-3]
                    R2unzip = R2[:-3]
                    !gunzip -c $R1 > $R1unzip
                    !gunzip -c $R2 > $R2unzip
                    print(segment)
                    mutCalling.sortFastq(R1=R1unzip, R2=R2unzip, segment=os.path.basename(segment),
                                        database = ampliconDesign, basename=baseName, 
                              fastaFiles = os.path.join(genomeFiles, os.path.basename(segment)),
                              matchLength = endMatching,
                                       plasmid = os.path.basename(experiment) == 'plasmid' )
                    !rm -f $R1unzip
                    !rm -f $R2unzip
    #zip the files
    zip = os.path.join(segment, '*.fastq')
    !gzip $zip
    
importlib.reload(mutCalling)
popList = []

for experiment in glob.glob(os.path.join(sequenceDirectory, '*')):
    for segment in glob.glob(os.path.join(experiment, '*')):
        popList += [(segment, overwrite)]
while len(popList) > 0:
    with Pool(processes = numProcessors) as p:
        p.map(analyzeSegment, [popList.pop() for i in range(numProcessors) if len(popList) != 0])


gzip: Sequencing/wholeGenome/MG/HA/*.fastq: No such file or directory
gzip: Sequencing/wholeGenome/MG/NA/*.fastq: No such file or directory
gzip: Sequencing/wholeGenome/MG/NP/*.fastq: No such file or directory
gzip: Sequencing/wholeGenome/MG/NS/*.fastq: No such file or directory
gzip: Sequencing/wholeGenome/MG/PA/*.fastq: No such file or directory
gzip: Sequencing/wholeGenome/MG/M/*.fastq: No such file or directory
gzip: Sequencing/wholeGenome/MG/PB1/*.fastq: No such file or directory
gzip: Sequencing/wholeGenome/MG/PB2/*.fastq: No such file or directory
gzip: Sequencing/wholeGenome/NS2/PA/*.fastq: No such file or directory
gzip: Sequencing/wholeGenome/NS2/HA/*.fastq: No such file or directory
gzip: Sequencing/wholeGenome/NS2/M/*.fastq: No such file or directory
gzip: Sequencing/wholeGenome/NS2/PB2/*.fastq: No such file or directory
gzip: Sequencing/wholeGenome/NS2/NP/*.fastq: No such file or directory
gzip: Sequencing/wholeGenome/NS2/NA/*.fastq: No such file or directory
gzip: Sequenc

## Sorting out caps

RACE sequencing is inherently more messy, including inappropriate internal priming, etc. We also observe cap sequences (or what look like cap sequences) mostly in the + strand, either ccRNA or template-switching between cRNA and mRNA. As these interfere with good alignments, and we likely should consider cRNA-derived and mRNA-derived sequences as somewhat different cases, we have gone ahead and seprated them out. We do the same for vRNA, but in this case it is likley mostly removing template switching artifacts or odd concatemers.

first we sort and remove capped + sense

In [ ]:
importlib.reload(mutCalling)



def cappedSeparatecRNA(files):
    TSOsequence = 'AAGCAGTGGTATCAACGCAGAGTACAT'
    database, sets, experiment, segment, sampleName = os.path.normpath(files).split(os.path.sep)
    if segment in ['PB2','PA','PB1','NA']:
        cRNAsequence = 'AGCGAAAGCAGG'
    else:
        cRNAsequence = 'AGCAAAAGCAGG'
    #need the Hoffman sequences. Original sequencing runs combined vRNA and cRNA, with the assumption of no issues with caps. There was template-switching to
    #the Hoffman sequences, which was excluded from original publication by removing "capped" sequences. As we now wish to understand those sequences,
    #we must remove it somehow here. It is quite odd it is there in the first place, as that event would fail to have the TSO for subsequent
    #amplification, I can only persume a double-TSO event, which was weirdly common.
    HoffAdapt = 'TCTCAGGG'
    allowedMismatchHoff = 1
    #lets only allow 4 nucleotides as a cap for uncapped. First nucleotide ambiguity appears derived from weird
    #TSO internal binding, as there is a strong preference for an alternate G there that seems inappropriate. Lets see if that fixes things by requiring just the three standard nucleotides
    uncapLimit = 3
    #ignoring the terminal 3 nucleotides of TSO, as it is NGG, and there can be multiple G's so minimum distance will be something like 3
    capLengthmin = 10
    #number of differences in TSO
    toleratedTSO = 5
    #only 12 nt of 5' end to try and reduce exluding interesting indels, so lets keep tolerated differeces to 4, partricularly as you frequently lose resolution at firstr base
    toleratedFive = 3
    if overwrite: 
        remove = files + '_Capped_*.fastq*'
        !rm -rf $remove
        remove = files + '_UnCapped_*.fastq*'
        !rm -rf $remove
    #still get rid of any intermediate files regardless
    remove = files[:-1] + '_Capped_*.fastq'
    !rm -rf $remove
    remove = files[:-1] + '_UnCapped_*.fastq'
    !rm -rf $remove
    R1 = files + '_R1.fastq.gz'
    R2 = files + '_R2.fastq.gz'
    R1unzip = R1[:-3]
    R2unzip = R2[:-3]
    R1unzip = R1[:-3]
    R2unzip = R2[:-3]
    !gunzip -c $R1 > $R1unzip
    !gunzip -c $R2 > $R2unzip
    print(files)
    trim3 = 0
    capReadNumber = 'R2'
    capRead = R2unzip
    pairedRead = R1unzip
    if segment in trim3prime['cRNA']:
        trim3 = trim3prime['cRNA'][segment]
    if experiment in specialcRNA:
        if segment in specialcRNA[experiment]:
            capReadNumber = 'R1'
            capRead = R1unzip
            pairedRead = R2unzip

            
    mutCalling.capSortFastqOneEnd(capReadNumber=capReadNumber, capRead=capRead, pairedRead=pairedRead, outfileCappedPrefix=files + '_Capped',\
                        outfileUnCappedPrefix=files + '_UnCapped' , capLength = capLengthmin,\
                        cRNAseq= cRNAsequence, adapterSeq=TSOsequence, toleratedfivePrime= toleratedFive, toleratedAdapter = toleratedTSO,\
                        outStatsPrefix = files,\
                       primerAdapter= HoffAdapt, primerAdapterLimit =allowedMismatchHoff,uncapLimit=uncapLimit, \
                                 trim3primeCapRead=trim3, trim3primepairedRead=trim3)
    
    reads = ['R1','R2']
    states = ['_UnCapped_','_Capped_']
    for read in reads:
        for state in states:
            file = files + state + read + '.fastq'
            !gzip $file
    !rm -f $R1unzip
    !rm -f $R2unzip
popList = []

for experiment in glob.glob(os.path.join(sequenceDirectory, '*')):
    for segment in glob.glob(os.path.join(experiment, '*')):
        files = list(set([os.path.join(segment, '_'.join(os.path.basename(file).split('_')[0:2])) for\
                    file in glob.glob(os.path.join(segment, '*cRNA*.fastq.gz'))]))
        files += list(set([os.path.join(segment, '_'.join(os.path.basename(file).split('_')[0:3])) for\
                    file in glob.glob(os.path.join(segment, '*MACS_cRNA*.fastq.gz'))]) )
        
        for file in files:
            write = True

            if not overwrite:
                if os.path.exists(file + '_Capped_R2.fastq.gz'):
                    write = False
            if os.path.basename(experiment) not in specialcRNA:
                if os.path.basename(file).split('_')[1] == 'MACS':
                    write = False
            elif os.path.basename(segment) not in specialcRNA[os.path.basename(experiment)]:
                if os.path.basename(file).split('_')[-2] == 'MACS':
                    write = False
            else:
                if os.path.basename(file).split('_')[-2] != 'MACS':
                    write = False
            if write:
                popList += [file]
while len(popList) > 0:

    with Pool(processes = numProcessors) as p:
        p.map(cappedSeparatecRNA, [popList.pop() for i in range(numProcessors) if len(popList) != 0])

now negative sense

In [6]:
importlib.reload(mutCalling)

def cappedSeparatevRNA(files):
    TSOsequence = 'AAGCAGTGGTATCAACGCAGAGTACAT'
    segment = os.path.split(os.path.split(files)[0])[1]
    
    vRNAsequence = 'AGTAGAAACAAG'
    database, sets, experiment, segment, sampleName = os.path.normpath(files).split(os.path.sep)
    trim3 = 0
    #need the Hoffman sequences. Original sequencing runs combined vRNA and cRNA, with the assumption of no issues with caps. There was template-switching to
    #the Hoffman sequences, which was excluded from original publication by removing "capped" sequences. As we now wish to understand those sequences,
    #we must remove it somehow here. It is quite odd it is there in the first place, as that event would fail to have the TSO for subsequent
    #amplification, I can only persume a double-TSO event, which was weirdly common.
    HoffAdapt = 'TCTCAGGG'
    allowedMismatchHoff = 1
    #lets only allow 4 nucleotides as a cap for uncapped. First nucleotide ambiguity appears derived from weird
    #TSO internal binding, as there is a strong preference for an alternate G there that seems inappropriate. Lets see if that fixes things by requiring just the three standard nucleotides
    uncapLimit = 3
    #ignoring the terminal 3 nucleotides of TSO, as it is NGG, and there can be multiple G's so minimum distance will be something like 3
    capLengthmin = 10
    #number of differences in TSO
    toleratedTSO = 5
    #only 12 nt of 5' end to try and reduce exluding interesting indels, so lets keep tolerated differeces to 4, partricularly as you frequently lose resolution at firstr base
    toleratedFive = 3
    if overwrite: 
        remove = files + '_Capped_*.fastq*'
        !rm -rf $remove
        remove = files + '_UnCapped_*.fastq*'
        !rm -rf $remove
    #still get rid of any intermediate files regardless
    remove = files + '_Capped_*.fastq'
    !rm -rf $remove
    remove = files + '_UnCapped_*.fastq'
    !rm -rf $remove
    R1 = files + '_R1.fastq.gz'
    R2 = files + '_R2.fastq.gz'
    R1unzip = R1[:-3]
    R2unzip = R2[:-3]
    basename = files
    R1unzip = R1[:-3]
    R2unzip = R2[:-3]
    !gunzip -c $R1 > $R1unzip
    !gunzip -c $R2 > $R2unzip
    print(files)
    mutCalling.capSortFastqOtherEnd(capReadNumber='R2', capRead=R2unzip, pairedRead=R1unzip, outfileCappedPrefix=basename + '_Capped',\
                        outfileUnCappedPrefix=basename + '_UnCapped' , capLength = capLengthmin,\
                        vRNAseq= vRNAsequence, adapterSeq=TSOsequence, toleratedfivePrime= toleratedFive, toleratedAdapter = toleratedTSO,\
                        outStatsPrefix = basename,\
                       primerAdapter= HoffAdapt, primerAdapterLimit =allowedMismatchHoff,uncapLimit=uncapLimit,
                                   trim3primeCapRead=trim3, trim3primepairedRead = trim3)
    
    reads = ['R1','R2']
    states = ['UnCapped_','Capped_']
    for read in reads:
        for state in states:
            file = files[:-1] + state + read + '.fastq'
            !gzip $file
    !rm -f $R1unzip
    !rm -f $R2unzip

popList = []
for experiment in glob.glob(os.path.join(sequenceDirectory, '*')):
    for segment in glob.glob(os.path.join(experiment, '*')):
        files = list(set([os.path.join(segment, '_'.join(os.path.basename(file).split('_')[0:2])) for\
                    file in glob.glob(os.path.join(segment, '*vRNA*.fastq.gz'))]))
        for file in files:
            if not overwrite:
                if not os.path.exists(file[:-1] + 'Capped_R2.fastq.gz'):
                    popList += [file]
            else:
                popList += files
while len(popList) > 0:
    with Pool(processes = numProcessors) as p:
        p.map(cappedSeparatevRNA, [popList.pop() for i in range(numProcessors) if len(popList) != 0])

## Alignment

Take all sequences and align to correct template.

In [13]:
for experiment in glob.glob(os.path.join(sequenceDirectory, '*')):
    for segment in glob.glob(os.path.join(experiment, '*')):
        files = glob.glob(os.path.join(segment, '*.fastq.gz'))
        samples = list(set(['_'.join(file.split('_')[0:3]) for file in files if ((file.split('_')[-2] != 'cRNA') and (file.split('_')[-2] != 'vRNA') and (file.split('_')[1] != 'MACS'))]))
        samples += list(set(['_'.join(file.split('_')[0:4]) for file in files if ((file.split('_')[1] == 'MACS') and (file.split('_')[-2] != 'cRNA'))]))
        for sample in samples:
            print(sample)
            template = os.path.join(genomeFiles, os.path.basename(segment), '_'.join(sample.split('_')[1:]))
            if (os.path.basename(template).split('_')[0] == 'vRNA') or (os.path.basename(template).split('_')[0] == 'cRNA') or (os.path.basename(template).split('_')[0] == 'MACS'):
                template = os.path.join(genomeFiles, os.path.basename(segment), '_'.join(sample.split('_')[1:-1]))
            R1 = sample + '_R1.fastq.gz'
            R2 = sample + '_R2.fastq.gz'
            R1unzip = R1[:-3]
            R2unzip = R2[:-3]
            outputFolder = os.path.join(alignmentDirectory, os.path.basename(experiment),os.path.basename(segment))
            if not os.path.isdir(outputFolder):
                os.mkdir(outputFolder)
            output = os.path.join(outputFolder,os.path.basename(sample)) + '_'
            !gunzip -c $R1 > $R1unzip
            !gunzip -c $R2 > $R2unzip
            capture = !STAR --runMode alignReads --runThreadN $numProcessors --genomeDir $template --readFilesIn $R1unzip $R2unzip --outFileNamePrefix $output --outSAMtype BAM Unsorted --alignIntronMax 1 --outFilterMismatchNmax 999 --alignEndsType EndToEnd --outFilterMismatchNoverReadLmax 0.02
            logFile =  output + 'terminal.txt'
            with open(logFile, 'w') as outfile:
                for line in capture:
                    outfile.write(line + '\n')
            !rm -f $R1unzip
            !rm -f $R2unzip


Sequencing/wholeGenome/plasmid/PA/2_plasmid_2
Sequencing/wholeGenome/plasmid/PA/3_plasmid_2
Sequencing/wholeGenome/plasmid/PA/1_plasmid_2


## Sequence assignment

First lets take each individual bamfile and call nucleotides at all positions that meet a certain score threshold. Output as a simple tsv. Go ahead and multiprocess to get it over in an expeditious fashion.

In [14]:
importlib.reload(mutCalling)

def SNPcall(file):
    outfile = os.path.join(os.path.dirname(file), '_'.join(os.path.basename(file).split('_')[:-1]) + '_SNP.tsv')
    print(outfile)
    mutCalling.positionalData(sortedBamfile=file, outfile=outfile, qvalCutoff= qvalueCutoff)

    
popList = []

for experiment in glob.glob(os.path.join(alignmentDirectory, 'plasmid')):
    for segment in glob.glob(os.path.join(experiment, 'PA')):
        popList += glob.glob(os.path.join(segment, '*plasmid_2*.bam'))


while len(popList) > 0:
    with Pool(processes = numProcessors) as p:
        p.map(SNPcall, [popList.pop() for i in range(numProcessors) if len(popList) != 0])
    
        
    


Results/wholeGenomeAlign/plasmid/PA/2_plasmid_2_SNP.tsvResults/wholeGenomeAlign/plasmid/PA/1_plasmid_2_SNP.tsv

Results/wholeGenomeAlign/plasmid/PA/3_plasmid_2_SNP.tsv


Now combine all sequencing. For RACE samples only use positions not covered by other sequences. Output a simple file containing nucleotides observed, including indels, what amplicon, and position in the template tested. Can correct for deletion junction in a seperate file later.

For now just ignore capped species, not sure what I want to do with those to be honest.

In [3]:
database = pd.read_csv(ampliconDesign, keep_default_na=False)
database.loc[database.template == 'vRNA','template'] = 'vRNA_UnCapped'
database.loc[database.template == 'cRNA', 'template'] = 'cRNA_UnCapped'
for experiment in glob.glob(os.path.join(alignmentDirectory, '*')):
    for segment in glob.glob(os.path.join(experiment, '*')):
        replicates = set(['_'.join(os.path.basename(file).split('_')[0]) for file in glob.glob(os.path.join(segment, '*.bam'))])
        for replicate in replicates:
            output = []
            files = [file for file in glob.glob(os.path.join(segment,replicate) + '_*_SNP.tsv') if os.path.basename(file).split('_')[-2] != 'Capped']
            chromosome = os.path.basename(segment)
            for file in files:
                tempFrame = pd.read_csv(file, sep='\t')
                if len(tempFrame) != 0:
                    tempFrame['template'] =  '_'.join(os.path.basename(file).split('_')[1:3])
                    tempFrame['segment'] = tempFrame.segment.str.split('_', expand=True)[0]
                    tempFrame = tempFrame.merge(database)
                    tempFrame = tempFrame.loc[tempFrame.position > tempFrame.ignore_5,]
                    tempFrame['position'] = tempFrame.position - 1 - tempFrame.ignore_5 + tempFrame.start_5
                    tempFrame = tempFrame.loc[tempFrame.position<= tempFrame.start_3,]
                    tempFrame = tempFrame.drop(columns = ['ignore_5','ignore_3','start_5','start_3','forward_primer'])
                    output += [tempFrame]
            if(len(output)!=0):
                output = pd.concat(output)
                #I want to only use RACE data when necessary as they tend to be worse quality than the simple amplicons.
                #as the amplicon templates will always sort higher due to alphabetical order, we can use this one-liner
                output = output.sort_values('template', ascending=False).drop_duplicates('position',  keep="last")
                outfile = os.path.join(segment,replicate) + '_allPositions.tsv'
                output.sort_values('position').to_csv(outfile, sep='\t', index=False)


Add a capped one

In [2]:
database = pd.read_csv(ampliconDesign, keep_default_na=False)
database.loc[database.template == 'vRNA','template'] = 'vRNA_Capped'
database.loc[database.template == 'cRNA', 'template'] = 'cRNA_Capped'
for experiment in glob.glob(os.path.join(alignmentDirectory, '*')):
    for segment in glob.glob(os.path.join(experiment, '*')):
        replicates = set(['_'.join(os.path.basename(file).split('_')[0]) for file in glob.glob(os.path.join(segment, '*.bam'))])
        for replicate in replicates:
            output = []
            files = [file for file in glob.glob(os.path.join(segment,replicate) + '_*_SNP.tsv') if os.path.basename(file).split('_')[-2] != 'UnCapped']
            chromosome = os.path.basename(segment)
            for file in files:
                tempFrame = pd.read_csv(file, sep='\t')
                if len(tempFrame) != 0:
                    tempFrame['template'] =  '_'.join(os.path.basename(file).split('_')[1:3])
                    tempFrame['segment'] = tempFrame.segment.str.split('_', expand=True)[0]
                    tempFrame = tempFrame.merge(database)
                    tempFrame = tempFrame.loc[tempFrame.position > tempFrame.ignore_5,]
                    tempFrame['position'] = tempFrame.position - 1 - tempFrame.ignore_5 + tempFrame.start_5
                    tempFrame = tempFrame.loc[tempFrame.position<= tempFrame.start_3,]
                    tempFrame = tempFrame.drop(columns = ['ignore_5','ignore_3','start_5','start_3','forward_primer'])
                    output += [tempFrame]
            if(len(output)!=0):
                output = pd.concat(output)
                #I want to only use RACE data when necessary as they tend to be worse quality than the simple amplicons.
                #as the amplicon templates will always sort higher due to alphabetical order, we can use this one-liner
                output = output.sort_values('template', ascending=False).drop_duplicates('position',  keep="last")
                outfile = os.path.join(segment,replicate) + '_allPositionsCapped.tsv'
                output.sort_values('position').to_csv(outfile, sep='\t', index=False)